# Valuation Engine Prototype

This notebook serves as the experimental environment for the `ValuationEngine`. It demonstrates how to train and evaluate the hybrid model using both financial metrics and NLP embeddings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Add src to path
sys.path.append(os.path.abspath("../src"))
from valuation import ValuationEngine

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Loading
We load the dataset processed by the `FeatureProcessor` (Gold Layer). The engine expects flattened NLP columns (`nlp_0`, `nlp_1`, etc.).

In [ ]:
data_path = "../data/processed/PUBLIC_embedded.parquet"

if os.path.exists(data_path):
    df = pd.read_parquet(data_path)
    print(f"Loaded {len(df)} companies with {len(df.columns)} columns.")
    display(df.head())
else:
    print(f"Data not found at {data_path}. Creating dummy data for demonstration.")
    # Creating dummy data with flattened NLP features
    n_samples = 100
    df = pd.DataFrame({
        "forwardPE": np.random.uniform(10, 40, n_samples),
        "ev_to_ebitda": np.random.uniform(5, 20, n_samples),
        "ebitda": np.random.uniform(100, 1000, n_samples),
        "total_cash": np.random.uniform(50, 200, n_samples),
        "total_debt": np.random.uniform(10, 100, n_samples),
        "enterprise_value": np.random.uniform(500, 5000, n_samples)
    })
    # Add 384 NLP features
    nlp_data = np.random.rand(n_samples, 384)
    nlp_df = pd.DataFrame(nlp_data, columns=[f"nlp_{i}" for i in range(384)])
    df = pd.concat([df, nlp_df], axis=1)
    display(df.head())

## 2. Model Training & Evaluation
We initialize the engine in `public` mode and prepare the data for the default target: `enterprise_value`.

In [ ]:
# Initialize engine in public mode
engine = ValuationEngine(mode="public", n_estimators=100)

# Prepare features (X) and target (y)
X, y = engine.prepare_data(df, target_col="enterprise_value")

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} samples with {X_train.shape[1]} features.")

# Train with progress bar
engine.train(X_train, y_train)

# Evaluate performance
metrics = engine.evaluate(X_test, y_test)

print("\nPerformance Metrics:")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

## 3. Visualization
Visualize the Predicted vs Actual Enterprise Values.

In [ ]:
y_pred_df = engine.predict(X_test)
y_pred = y_pred_df["enterprise_value"]

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test["enterprise_value"], y=y_pred, alpha=0.6)
plt.plot([y_test.min().min(), y_test.max().max()], [y_test.min().min(), y_test.max().max()], 'r--', lw=2)
plt.xlabel("Actual Enterprise Value")
plt.ylabel("Predicted Enterprise Value")
plt.title("Actual vs Predicted Valuation")
plt.show()